# 🎬 Reels — Free-GPU Talking Avatar (Kaggle / MuseTalk)

Enable **Settings → Accelerator → GPU T4 x2** and **Internet: On**, then run top to bottom.

Output: a 9:16 `reel.mp4` you download from the Kaggle output panel.


In [ ]:
# 1) GPU check
!nvidia-smi -L
import torch;print('CUDA:',torch.cuda.is_available())

In [ ]:
# 2) Get repo
%cd /kaggle/working
!git clone https://github.com/amsinghnavdeep/reels.git 2>/dev/null || (cd reels && git pull)
%cd /kaggle/working/reels
!pip -q install edge-tts pydub pyyaml

In [ ]:
# 3) Clone + install MuseTalk (one-time)
import os
os.makedirs('engines',exist_ok=True)
if not os.path.isdir('engines/MuseTalk'):
    !git clone https://github.com/TMElyralab/MuseTalk.git engines/MuseTalk
%cd /kaggle/working/reels/engines/MuseTalk
!pip -q install -r requirements.txt
!pip -q install --no-cache-dir -U openmim
!mim install mmengine 'mmcv==2.0.1' 'mmdet==3.1.0' 'mmpose==1.1.0'
!mkdir -p ffmpeg && cd ffmpeg && wget -q https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz && tar xf ffmpeg-release-amd64-static.tar.xz --strip-components=1
os.environ['FFMPEG_PATH']='/kaggle/working/reels/engines/MuseTalk/ffmpeg'

In [ ]:
# 4) Download weights (~5 GB).
#    MuseTalk's download_weights.sh uses the removed `huggingface-cli` (downloads
#    nothing); use the huggingface_hub Python API instead.
%cd /kaggle/working/reels/engines/MuseTalk
!curl -sL https://raw.githubusercontent.com/amsinghnavdeep/reels/main/notebooks/dl_weights.py -o /tmp/dl.py && python3 /tmp/dl.py

In [ ]:
# 5) Driving VIDEO + cloned voice. Add your files via 'Add Data' or upload to /kaggle/working,
#    then set DRIVE_VIDEO / VOICE_SRC to their paths. Real footage = sharp mouth + real motion.
%cd /kaggle/working/reels
import os, subprocess
os.makedirs('output', exist_ok=True)
DRIVE_VIDEO = '/kaggle/input/YOUR-DATASET/your_talking_head.mp4'   # <-- set this
VOICE_SRC   = '/kaggle/input/YOUR-DATASET/your_voice.m4a'          # <-- set this
SCRIPT = 'examples/script.txt'   # edit with your words

# Optional: trim/crop out burned captions. e.g. TRIM='-ss 15 -to 19.5'  CROP='crop=360:360:0:48'
TRIM = ''
CROP = ''
# ALWAYS normalise to a safe path: spaces / # / () in the name break MuseTalk's frame extraction.
# cap the long side at 1280px so a 4K clip doesn't exhaust RAM (MuseTalk loads every frame).
_dn = "scale=1280:1280:force_original_aspect_ratio=decrease:force_divisible_by=2"
vf = f'-vf "{CROP},{_dn}"' if CROP else f'-vf "{_dn}"'
subprocess.run(f'ffmpeg -y {TRIM} -i "{DRIVE_VIDEO}" {vf} -an -r 25 -c:v libx264 -pix_fmt yuv420p output/drive_clip.mp4', shell=True, check=True)
DRIVE_VIDEO = 'output/drive_clip.mp4'
print('driving video:', DRIVE_VIDEO)
subprocess.run(f'ffmpeg -y -i "{VOICE_SRC}" -ar 16000 -ac 1 output/voice_ref.wav', shell=True, check=True)
!pip -q install coqui-tts
!COQUI_TOS_AGREED=1 python tts.py --script $SCRIPT --engine xtts --clone output/voice_ref.wav --lang hi --output output/speech.wav
#   Or a clean non-cloned Hindi voice:  !python tts.py --script $SCRIPT --engine edge --voice hi-IN-MadhurNeural --output output/speech.wav
from IPython.display import Audio; Audio('output/speech.wav')# normalise the voice sample to 16k mono wav
subprocess.run(f'ffmpeg -y -i "{VOICE_SRC}" -ar 16000 -ac 1 /kaggle/working/reels/output/voice_ref.wav', shell=True, check=True)

# clone your voice with XTTS-v2 in an ISOLATED py3.10 env
#   (Colab's Python 3.12 kernel + numpy 2 break coqui-tts; py3.10 is the supported combo)
!source /opt/conda/etc/profile.d/conda.sh && \
  (conda env list | grep -q '/envs/tts$' || conda create -y -q -n tts python=3.10) && \
  conda activate tts && \
  export MPLBACKEND=Agg && \
  pip -q install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118 && \
  pip -q install transformers==4.35.2 && \
  pip -q install coqui-tts==0.22.1 && \
  COQUI_TOS_AGREED=1 python /kaggle/working/reels/tts.py --script /kaggle/working/reels/examples/script.txt \
    --engine xtts --clone /kaggle/working/reels/output/voice_ref.wav --lang hi \
    --output /kaggle/working/reels/output/speech_raw.wav
#   Prefer a clean non-cloned Hindi voice instead? (no isolated env needed)
#   !python tts.py --script "$SCRIPT" --engine edge --voice hi-IN-MadhurNeural --output output/speech_raw.wav
import os; assert os.path.exists('output/speech_raw.wav'), 'TTS failed \u2014 see the logs above'
# TIMING FIX: strip leading/trailing silence so the mouth starts moving exactly with the audio
subprocess.run('ffmpeg -y -i output/speech_raw.wav -af '
  '"silenceremove=start_periods=1:start_silence=0.05:start_threshold=-45dB:'
  'stop_periods=-1:stop_silence=0.2:stop_threshold=-45dB" '
  '-ar 16000 -ac 1 output/speech.wav', shell=True, check=True)
from IPython.display import Audio; Audio(filename='output/speech.wav')


In [ ]:
# 6) MuseTalk lip-sync
import yaml,os
os.chdir('/kaggle/working/reels/engines/MuseTalk')
cfg={'task_0':{'video_path':('/kaggle/working/reels/'+DRIVE_VIDEO) if not DRIVE_VIDEO.startswith('/') else DRIVE_VIDEO,'audio_path':'/kaggle/working/reels/output/speech.wav'}}
os.makedirs('configs/inference',exist_ok=True)
open('configs/inference/reels.yaml','w').write(yaml.dump(cfg))
# guard MuseTalk's unconditional temp cleanup crash on still-image avatar input
!sed -i 's|^            shutil.rmtree(save_dir_full)|            if get_file_type(video_path) == "video": shutil.rmtree(save_dir_full)|' scripts/inference.py
!python -m scripts.inference --inference_config configs/inference/reels.yaml --result_dir /kaggle/working/reels/output/muse --unet_model_path models/musetalkV15/unet.pth --unet_config models/musetalkV15/musetalk.json --version v15 --fps 25 --batch_size 4 --use_float16 --parsing_mode jaw --extra_margin 10

In [ ]:
# 6b) SHARPEN — restore the soft MuseTalk mouth patch back to input-quality with GFPGAN
#     (face restore) + Real-ESRGAN (frame upscaler). Runs in the same py3.10 'muse' env.
import os
os.chdir('/content/reels/engines/MuseTalk') if os.path.isdir('/content/reels') else os.chdir('/kaggle/working/reels/engines/MuseTalk')
RAW = None
import glob
cands = sorted(glob.glob('/kaggle/working/reels/output/muse/**/*.mp4', recursive=True), key=os.path.getmtime)
assert cands, 'No MuseTalk output found \u2014 run cell 6 first.'
RAW = cands[-1]; print('raw MuseTalk clip:', RAW)

# one-time: install GFPGAN + Real-ESRGAN into the muse env. basicsr imports
# torchvision.transforms.functional_tensor, removed in torchvision 0.16 -> patch to functional.
!source /opt/conda/etc/profile.d/conda.sh && conda activate muse && \
  pip -q install gfpgan realesrgan basicsr facexlib && \
  BS=$(/opt/conda/envs/muse/bin/python -c "import basicsr,os;print(os.path.dirname(basicsr.__file__))") && \
  sed -i 's/torchvision.transforms.functional_tensor/torchvision.transforms.functional/' \
    $BS/data/degradations.py 2>/dev/null; echo patched

# split MuseTalk clip into frames, GFPGAN-restore each (with Real-ESRGAN background), reassemble + mux audio.
os.makedirs('/kaggle/working/reels/output/enh_in', exist_ok=True)
os.makedirs('/kaggle/working/reels/output/enh_out', exist_ok=True)
!rm -f /kaggle/working/reels/output/enh_in/* /kaggle/working/reels/output/enh_out/*
!ffmpeg -y -i "$RAW" -qscale:v 1 /kaggle/working/reels/output/enh_in/f%06d.png
# GFPGAN v1.4 restores the face (mouth/teeth) at 2x; we fit back to 1080x1920 in cell 7.
# write the restore script to a file (heredocs are unreliable inside a '!' cell)
open('/kaggle/working/reels/output/_restore.py', 'w').write(r'''
import os, glob, cv2
from gfpgan import GFPGANer
root = "/kaggle/working/reels"
gfp = GFPGANer(model_path="https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth",
               upscale=2, arch="clean", channel_multiplier=2, bg_upsampler=None)
frames = sorted(glob.glob(root + "/output/enh_in/*.png"))
for i, fp in enumerate(frames):
    img = cv2.imread(fp, cv2.IMREAD_COLOR)
    _, _, out = gfp.enhance(img, has_aligned=False, only_center_face=True, paste_back=True)
    cv2.imwrite(root + "/output/enh_out/" + os.path.basename(fp), out)
    if i % 50 == 0: print("  restored", i, "/", len(frames))
print("done", len(frames), "frames")
''')
!source /opt/conda/etc/profile.d/conda.sh && conda activate muse && \
  MPLBACKEND=Agg python /kaggle/working/reels/output/_restore.py
# reassemble restored frames at 25fps and mux the cloned audio
!ffmpeg -y -framerate 25 -i /kaggle/working/reels/output/enh_out/f%06d.png -i /kaggle/working/reels/output/speech.wav \
  -c:v libx264 -pix_fmt yuv420p -c:a aac -shortest /kaggle/working/reels/output/muse_hd.mp4
print('enhanced clip:', '/kaggle/working/reels/output/muse_hd.mp4')


In [ ]:
# 7) Compose a 9:16 reel + preview/download (prefers the GFPGAN-restored clip)
import os; os.chdir('/kaggle/working/reels')
import glob, os
hd = 'output/muse_hd.mp4'
if os.path.exists(hd):
    raw = hd
else:
    clips = sorted(glob.glob('output/muse/**/*.mp4', recursive=True), key=os.path.getmtime)
    assert clips, 'No MuseTalk output found \u2014 check the previous cell logs.'
    raw = clips[-1]
print('composing from:', raw)
!python reel_utils.py --in "$raw" --out output/reel.mp4
print('reel written to /kaggle/working/reels/output/reel.mp4 \u2014 download from the Output panel')
